In [9]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers.csv
Saving revenue.csv to revenue.csv
Saving subscriptions.csv to subscriptions.csv


In [18]:
import pandas as pd

customers = pd.read_csv('customers.csv')
subscriptions = pd.read_csv('subscriptions.csv')
revenue = pd.read_csv('revenue.csv')

print("Customers shape: ", customers.shape)
print("Subscriptions shape:", subscriptions.shape)
print("Revenue shape:" , revenue.shape)

customers.head()


Customers shape:  (1000, 6)
Subscriptions shape: (988, 4)
Revenue shape: (988, 6)


,customer_id,signup_date,plan_type,monthly_fee,acquisition_cost,churn_date
0,1001,2024-11-07,Basic,50,30,NaN
1,1002,2024-06-06,Basic,50,30,NaN
2,1003,2024-12-31,Basic,50,30,NaN
3,1004,2024-11-21,Pro,200,100,NaN
4,1005,2024-08-16,Pro,200,100,NaN


In [19]:
print(customers.isnull().sum())
print(customers.dtypes)

customer_id           0
signup_date           0
plan_type             0
monthly_fee           0
acquisition_cost      0
churn_date          832
dtype: int64
customer_id          int64
signup_date         object
plan_type           object
monthly_fee          int64
acquisition_cost     int64
churn_date          object
dtype: object


In [20]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])

subscriptions['month'] = pd.to_datetime(subscriptions['month'])
revenue['month'] = pd.to_datetime(revenue['month'])

customers.dtypes

,0
customer_id,int64
signup_date,datetime64[ns]
plan_type,object
monthly_fee,int64
acquisition_cost,int64
churn_date,datetime64[ns]


In [21]:
customers['cohort_month'] = customers['signup_date'].dt.to_period('M')
customers[['customer_id', 'signup_date' , 'cohort_month']].head()

,customer_id,signup_date,cohort_month
0,1001,2024-11-07,2024-11
1,1002,2024-06-06,2024-06
2,1003,2024-12-31,2024-12
3,1004,2024-11-21,2024-11
4,1005,2024-08-16,2024-08


In [22]:
merged = subscriptions.merge(customers[['customer_id', 'cohort_month']], on='customer_id')
merged['activity_month'] = merged['month'].dt.to_period('M')
merged['months_since_signup'] = (merged['activity_month'] - merged['cohort_month']).apply(lambda x: x.n)

merged.head()

,subscription_id,customer_id,month,monthly_fee,cohort_month,activity_month,months_since_signup
0,S-1020-202410,1020,2024-10-01,200,2024-10,2024-10,0
1,S-1020-202411,1020,2024-11-01,200,2024-10,2024-11,1
2,S-1020-202412,1020,2024-12-01,200,2024-10,2024-12,2
3,S-1020-202501,1020,2025-01-01,200,2024-10,2025-01,3
4,S-1020-202502,1020,2025-02-01,200,2024-10,2025-02,4


In [23]:
cohort_data = merged.groupby(['cohort_month', 'months_since_signup'])['customer_id'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='months_since_signup', values='customer_id')

cohort_pivot

months_since_signup,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
cohort_month,,,,,,,,,,,,,,,,,
2024-01,10.0,9.0,9.0,9.0,9.0,7.0,7.0,7.0,7.0,7.0,6.0,5.0,4.0,3.0,2.0,1.0,1.0
2024-02,8.0,8.0,7.0,7.0,7.0,6.0,6.0,6.0,6.0,5.0,5.0,5.0,5.0,4.0,1.0,1.0,NaN
2024-03,13.0,13.0,13.0,11.0,9.0,9.0,9.0,8.0,7.0,6.0,6.0,6.0,6.0,6.0,5.0,2.0,NaN
2024-04,10.0,10.0,8.0,8.0,7.0,7.0,7.0,6.0,6.0,5.0,4.0,4.0,2.0,NaN,NaN,NaN,NaN
2024-05,10.0,10.0,10.0,8.0,6.0,6.0,5.0,5.0,4.0,4.0,4.0,3.0,2.0,NaN,NaN,NaN,NaN
2024-06,10.0,10.0,10.0,9.0,9.0,7.0,5.0,5.0,5.0,3.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN
2024-07,10.0,10.0,6.0,5.0,5.0,5.0,5.0,5.0,4.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,8.0,8.0,7.0,6.0,5.0,4.0,4.0,2.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,6.0,6.0,5.0,3.0,3.0,3.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
cohort_size = cohort_pivot.iloc[:, 0]
retention_pct = cohort_pivot.divide(cohort_size, axis=0) * 100
retention_pct.round(1)

months_since_signup,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
cohort_month,,,,,,,,,,,,,,,,,
2024-01,100.0,90.0,90.0,90.0,90.0,70.0,70.0,70.0,70.0,70.0,60.0,50.0,40.0,30.0,20.0,10.0,10.0
2024-02,100.0,100.0,87.5,87.5,87.5,75.0,75.0,75.0,75.0,62.5,62.5,62.5,62.5,50.0,12.5,12.5,NaN
2024-03,100.0,100.0,100.0,84.6,69.2,69.2,69.2,61.5,53.8,46.2,46.2,46.2,46.2,46.2,38.5,15.4,NaN
2024-04,100.0,100.0,80.0,80.0,70.0,70.0,70.0,60.0,60.0,50.0,40.0,40.0,20.0,NaN,NaN,NaN,NaN
2024-05,100.0,100.0,100.0,80.0,60.0,60.0,50.0,50.0,40.0,40.0,40.0,30.0,20.0,NaN,NaN,NaN,NaN
2024-06,100.0,100.0,100.0,90.0,90.0,70.0,50.0,50.0,50.0,30.0,10.0,10.0,NaN,NaN,NaN,NaN,NaN
2024-07,100.0,100.0,60.0,50.0,50.0,50.0,50.0,50.0,40.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,100.0,100.0,87.5,75.0,62.5,50.0,50.0,25.0,12.5,12.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,100.0,100.0,83.3,50.0,50.0,50.0,33.3,16.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
mrr_trend = revenue.groupby(revenue['month'].dt.to_period('M'))['amount'].sum().reset_index()
mrr_trend.columns = ['month', 'total_mrr']
mrr_trend

,month,total_mrr
0,2024-01,1850
1,2024-02,4600
2,2024-03,7650
3,2024-04,9600
4,2024-05,11750
5,2024-06,12700
6,2024-07,15900
7,2024-08,17800
8,2024-09,17150
9,2024-10,18750


In [26]:
revenue_by_plan = merged.merge(customers[['customer_id', 'plan_type']], on='customer_id').groupby('plan_type')['monthly_fee'].sum().reset_index()
revenue_by_plan.columns = ['plan_type', 'total revenue']
revenue_by_plan

,plan_type,total revenue
0,Basic,16000
1,Enterprise,167000
2,Pro,66800


In [27]:
avg_revenue_per_customer = merged.merge(customers[['customer_id', 'plan_type']], on= 'customer_id').groupby('plan_type')['monthly_fee'].sum() / customers.groupby('plan_type')['customer_id'].nunique()
avg_cac = customers.groupby('plan_type')['acquisition_cost']. mean()

ltv_cac = pd.DataFrame({'avg_ltv':avg_revenue_per_customer, 'avg_cac': avg_cac})
ltv_cac['ltv_cac_ratio'] = ltv_cac['avg_ltv'] / ltv_cac['avg_cac']
ltv_cac

,avg_ltv,avg_cac,ltv_cac_ratio
plan_type,,,
Basic,48.929664,30.0,1.630989
Enterprise,506.060606,200.0,2.530303
Pro,194.752187,100.0,1.947522


In [31]:
customers_sql = customers.copy()
customers_sql['cohort_month'] = customers_sql['cohort_month'].astype(str)

merged_sql = merged.copy()
merged_sql['cohort_month'] = merged_sql['cohort_month'].astype(str)
merged_sql['activity_month'] = merged_sql['activity_month'].astype(str)

import sqlite3
conn = sqlite3.connect('saas_cohort.db')

customers_sql.to_sql('customers', conn, if_exists='replace', index=False)
subscriptions.to_sql('subscriptions', conn, if_exists='replace', index=False)
revenue.to_sql('revenue', conn, if_exists='replace', index=False)
merged_sql.to_sql('merged', conn, if_exists='replace', index=False)

print("Databse created successfully")


Databse created successfully


In [32]:
query = """
SELECT cohort_month, months_since_signup, COUNT(DISTINCT customer_id) as active_customers
FROM merged
GROUP BY cohort_month, months_since_signup
ORDER BY cohort_month, months_since_signup
"""
result =pd.read_sql_query(query, conn)
result.head(15)

,cohort_month,months_since_signup,active_customers
0,2024-01,0,10
1,2024-01,1,9
2,2024-01,2,9
3,2024-01,3,9
4,2024-01,4,9
5,2024-01,5,7
6,2024-01,6,7
7,2024-01,7,7
8,2024-01,8,7
9,2024-01,9,7


In [33]:
query2 = """
SELECT strftime('%Y-%m', month) as month, SUM(amount) as total_mrr
FROM revenue
GROUP BY month
ORDER BY month
"""
mrr_sql = pd.read_sql_query(query2, conn)
mrr_sql


,month,total_mrr
0,2024-01,1850
1,2024-02,4600
2,2024-03,7650
3,2024-04,9600
4,2024-05,11750
5,2024-06,12700
6,2024-07,15900
7,2024-08,17800
8,2024-09,17150
9,2024-10,18750


In [35]:
query2 = """
SELECT strftime('%Y-%m', month) as month, SUM(amount) as total_mrr
FROM revenue
GROUP BY month
ORDER BY month
"""
mrr_sql = pd.read_sql_query(query2, conn)
mrr_sql


,month,total_mrr
0,2024-01,1850
1,2024-02,4600
2,2024-03,7650
3,2024-04,9600
4,2024-05,11750
5,2024-06,12700
6,2024-07,15900
7,2024-08,17800
8,2024-09,17150
9,2024-10,18750


In [42]:
retention_pct.to_csv('retention_pct.csv')
mrr_trend_export = mrr_trend.copy()
mrr_trend_export['month'] = mrr_trend_export['month'].astype(str)
mrr_trend_export.to_csv('mrr_trend.csv', index=False)
revenue_by_plan.to_csv('revenue_by_plan.csv', index=False)
ltv_cac.to_csv('ltv_cac.csv')

from google.colab import files
files.download('retention_pct.csv')
files.download('mrr_trend.csv')
files.download('revenue_by_plan.csv')
files.download('ltv_cac.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
print(type(mrr_trend))

<class 'pandas.core.frame.DataFrame'>
